# Differential Calculus for Machine Learning

Distilled from Géron's [math_differential_calculus.ipynb](https://github.com/ageron/handson-ml3/blob/main/math_differential_calculus.ipynb). Only the parts ML actually uses. Proof-heavy material, Taylor series, Hessians, integrals — deferred.

**Goal:** after this notebook you should be able to:
- Read a derivative and see the *slope of a tangent line* in your head.
- Read a gradient and see an *arrow pointing uphill on a surface*.
- Explain why **gradient descent** is the engine behind every ML training loop.

The emphasis is: **equation → picture → what the ML algorithm is doing with it.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

## 1. Why does ML need calculus?

Training a model is a **minimization problem**:

```
find weights w that minimize  L(w) = loss over the training data
```

There are billions of possible `w` vectors — you can't try them all. Calculus tells us which direction to move `w` to reduce `L` fastest. That single idea — **the gradient gives us the direction of improvement** — powers linear regression, logistic regression, SVMs, and every neural network.

So: calculus is not for deriving formulas by hand. It's for understanding *why the training loop works at all.*

## 2. Derivatives — the slope of a curve

The derivative of `f(x)` at a point `x₀` is:

```
f'(x₀) = lim  (f(x₀ + h) − f(x₀)) / h
         h→0
```

Read it as: **"the slope of the tangent line to `f` at `x₀`."**

- `f'(x₀) > 0` → function is increasing there. Step right to go up, left to go down.
- `f'(x₀) < 0` → function is decreasing. Step right to go down.
- `f'(x₀) = 0` → flat spot. Could be a minimum, maximum, or saddle.

**This last fact is the hinge of ML.** To minimize a loss, find where its derivative is zero (or walk until you get there).

In [ ]:
# Visualize: the derivative IS the slope of the tangent line
def f(x):
    return x**2

def f_prime(x):
    return 2 * x

xs = np.linspace(-3, 3, 200)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, x0 in zip(axes, [-2.0, 0.0, 1.5]):
    slope = f_prime(x0)
    tangent = f(x0) + slope * (xs - x0)
    ax.plot(xs, f(xs), 'C0', label='f(x) = x²')
    ax.plot(xs, tangent, 'C3--', label=f"tangent, slope = {slope:.1f}")
    ax.plot(x0, f(x0), 'ko')
    ax.set_title(f"x₀ = {x0}   f'(x₀) = {slope}")
    ax.set_ylim(-2, 10); ax.grid(alpha=0.3); ax.legend()

plt.tight_layout(); plt.show()

### Numerical derivative (the definition, in code)

You almost never compute derivatives by hand in ML — autograd does it. But knowing the limit definition as code demystifies the symbol.

In [ ]:
def numerical_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)   # 'central' form — more accurate

for x0 in [-2.0, 0.0, 1.5, 3.0]:
    exact = f_prime(x0)
    approx = numerical_derivative(f, x0)
    print(f'x={x0:>4}   exact={exact:>6.3f}   numerical={approx:>8.5f}')

## 3. Rules worth recognizing (don't memorize)

You rarely compute these by hand. Just recognize the patterns when they appear in a formula.

| Function | Derivative | Why it matters in ML |
|---------|------------|----------------------|
| `c` (constant) | `0` | bias terms vanish when differentiating w.r.t. `w` |
| `xⁿ` | `n · xⁿ⁻¹` | core polynomial rule; used in MSE loss |
| `eˣ` | `eˣ` | softmax, sigmoid derivations |
| `ln(x)` | `1 / x` | cross-entropy / log-loss |
| `sin, cos` | `cos, −sin` | rarely in classical ML; periodic features |

**Combining rules:**
- Sum: `(f + g)' = f' + g'`
- Product: `(f·g)' = f'g + fg'`
- Chain: `(f(g(x)))' = f'(g(x)) · g'(x)` ← **this is the one backprop uses**

## 4. Partial derivatives — many inputs, one output

Loss functions don't take one number — they take a vector `w = (w₁, w₂, ..., wₙ)`. A **partial derivative** asks: *if I nudge only `w_i` while freezing the others, how does `L` change?*

```
∂L/∂w_i = lim  (L(..., w_i + h, ...) − L(..., w_i, ...)) / h
          h→0
```

Read it as: **"slope of `L` in the `w_i` direction, all other axes fixed."**

For MNIST's 784-weight classifier, there are 784 partial derivatives — one per weight. Together they tell us how sensitive the loss is to each individual pixel's weight.

In [ ]:
# Visualize a 2D surface and the two 1D 'slices' that give partial derivatives
def L(w1, w2):
    return w1**2 + 3 * w2**2

w1_vals = np.linspace(-3, 3, 60)
w2_vals = np.linspace(-3, 3, 60)
W1, W2 = np.meshgrid(w1_vals, w2_vals)
Z = L(W1, W2)

fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(1, 3, 1, projection='3d')
ax1.plot_surface(W1, W2, Z, cmap='viridis', alpha=0.7)
ax1.set_title('Loss surface L(w₁, w₂) = w₁² + 3w₂²')
ax1.set_xlabel('w₁'); ax1.set_ylabel('w₂'); ax1.set_zlabel('L')

w2_fixed = 1.0
ax2 = fig.add_subplot(1, 3, 2)
ax2.plot(w1_vals, L(w1_vals, w2_fixed), 'C0')
ax2.set_title(f'Slice at w₂ = {w2_fixed}\n∂L/∂w₁ is the slope here')
ax2.set_xlabel('w₁'); ax2.set_ylabel('L'); ax2.grid(alpha=0.3)

w1_fixed = 1.0
ax3 = fig.add_subplot(1, 3, 3)
ax3.plot(w2_vals, L(w1_fixed, w2_vals), 'C3')
ax3.set_title(f'Slice at w₁ = {w1_fixed}\n∂L/∂w₂ is the slope here')
ax3.set_xlabel('w₂'); ax3.set_ylabel('L'); ax3.grid(alpha=0.3)

plt.tight_layout(); plt.show()

For `L(w₁, w₂) = w₁² + 3w₂²`:
```
∂L/∂w₁ = 2·w₁       (treat w₂ as a constant)
∂L/∂w₂ = 6·w₂       (treat w₁ as a constant)
```

Each partial is a regular 1D derivative taken along one axis.

## 5. The gradient — the single most important object in ML

The **gradient** bundles all partial derivatives into one vector:

```
∇L(w) = [ ∂L/∂w₁,  ∂L/∂w₂,  ...,  ∂L/∂wₙ ]
```

Two geometric facts to burn in:

1. **`∇L(w)` points in the direction of steepest *increase* of `L`.** Its length is how steep.
2. **`−∇L(w)` points in the direction of steepest *decrease*.** ← This is the direction SGD moves.

So for ML: compute gradient → step in the opposite direction → loss goes down. That's literally the algorithm.

In [ ]:
# Contour plot with gradient arrows overlaid
def L(w):
    return w[0]**2 + 3 * w[1]**2

def grad_L(w):
    return np.array([2 * w[0], 6 * w[1]])

w1 = np.linspace(-3, 3, 100)
w2 = np.linspace(-3, 3, 100)
W1, W2 = np.meshgrid(w1, w2)
Z = W1**2 + 3 * W2**2

# Sample some points, draw gradient vectors
sample = [np.array([p, q]) for p in [-2, -1, 0, 1, 2] for q in [-2, -1, 0, 1, 2] if not (p == 0 and q == 0)]

plt.figure(figsize=(7, 6))
plt.contour(W1, W2, Z, levels=15, cmap='viridis', alpha=0.6)
for p in sample:
    g = grad_L(p)
    # Scale arrows so they're visible (not too long)
    g_scaled = g / (np.linalg.norm(g) + 1e-8) * 0.4
    plt.quiver(p[0], p[1], g_scaled[0], g_scaled[1], angles='xy', scale_units='xy', scale=1, color='C3')
plt.plot(0, 0, 'g*', markersize=20, label='minimum')
plt.title('Gradient vectors point UPHILL (away from the minimum)')
plt.xlabel('w₁'); plt.ylabel('w₂')
plt.grid(alpha=0.3); plt.legend(); plt.gca().set_aspect('equal')
plt.show()

Notice how every red arrow points *away* from the green star (the minimum). Gradient descent simply walks in the reverse direction.

## 6. Gradient descent — connecting calculus to the training loop

The whole training algorithm is three lines:

```
initialize w randomly
for each step:
    w ← w − η · ∇L(w)
```

Unpack the update:
- `∇L(w)` — steepest uphill direction (calculus computes this)
- `−∇L(w)` — steepest downhill direction
- `η` (learning rate) — how big a step to take
- `w ← w − η · ∇L(w)` — take that step (linear algebra does this)

**Pure calculus tells you *which way to step*. Pure linear algebra carries out the step.** That's how the two math pillars combine.

In [ ]:
# Visualize gradient descent trajectories from several starting points
def run_gd(w0, eta, steps=30):
    w = w0.copy()
    path = [w.copy()]
    for _ in range(steps):
        w = w - eta * grad_L(w)
        path.append(w.copy())
    return np.array(path)

plt.figure(figsize=(7, 6))
plt.contour(W1, W2, Z, levels=15, cmap='viridis', alpha=0.5)

starts = [np.array([2.5, 2.5]), np.array([-2.8, 1.5]), np.array([1.0, -2.5]), np.array([-2.0, -2.5])]
for w0 in starts:
    path = run_gd(w0, eta=0.1)
    plt.plot(path[:, 0], path[:, 1], 'o-', markersize=3, label=f'start {w0.tolist()}')

plt.plot(0, 0, 'g*', markersize=20)
plt.title('Gradient descent from multiple starts — all converge to the minimum')
plt.xlabel('w₁'); plt.ylabel('w₂')
plt.grid(alpha=0.3); plt.legend(fontsize=8); plt.gca().set_aspect('equal')
plt.show()

### Learning rate — the single hyperparameter to understand first

- Too small → training crawls.
- Just right → smooth descent.
- Too large → overshoots, may diverge (loss blows up).

Every tuning guide in the book eventually tells you to plot the loss curve and adjust `η`. The picture below shows why.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, eta, title in zip(axes,
                          [0.02, 0.15, 0.34],
                          ['η = 0.02 (too small)', 'η = 0.15 (good)', 'η = 0.34 (too large)']):
    path = run_gd(np.array([2.8, 2.5]), eta=eta, steps=30)
    ax.contour(W1, W2, Z, levels=15, cmap='viridis', alpha=0.5)
    ax.plot(path[:, 0], path[:, 1], 'ro-', markersize=3)
    ax.plot(0, 0, 'g*', markersize=15)
    ax.set_title(title)
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
    ax.grid(alpha=0.3); ax.set_aspect('equal')

plt.tight_layout(); plt.show()

## 7. The chain rule — how backprop works

Most ML loss functions are **compositions**:

```
loss = compare( model(w, x),  y )
```

The chain rule says: if `y = f(g(x))`, then:

```
dy/dx = f'(g(x)) · g'(x)
```

In words: *"multiply the local slopes along the chain."*

**ML mapping:**
- Neural networks = long chains of function compositions.
- Backpropagation = the chain rule applied efficiently to every layer, right-to-left.
- Each parameter's gradient is a *product of local slopes* from its layer forward to the loss.

In [ ]:
# Concrete example: y = (3x + 1)²  — a composition of two functions
#   outer: f(u) = u²     → f'(u) = 2u
#   inner: g(x) = 3x + 1 → g'(x) = 3
#   dy/dx = f'(g(x)) · g'(x) = 2(3x + 1) · 3 = 6(3x + 1)

def y_fn(x):     return (3 * x + 1) ** 2
def y_prime(x):  return 6 * (3 * x + 1)

for x0 in [-1, 0, 1, 2]:
    exact = y_prime(x0)
    numeric = numerical_derivative(y_fn, x0)
    print(f'x={x0}  exact={exact:>5}   numerical={numeric:>8.3f}')

For now, chain rule intuition is enough. The full backprop derivation lives in Ch. 10.

## 8. Minima, maxima, and why convexity matters

Where `∇L(w) = 0` is called a **critical point**. Three flavors:

| Shape | At critical point | ML meaning |
|-------|-------------------|------------|
| Bowl (convex up) | local minimum | ✅ GD converges here |
| Dome (concave) | local maximum | GD moves *away* — not a concern for loss minimization |
| Saddle | flat in some directions, curved in others | GD can stall here (neural networks) |

**Convex** means: the surface is shaped like a single bowl — any line segment between two points on the graph lies above the graph.
- Linear regression (MSE), logistic regression, SVMs → convex. **GD reaches the global min.**
- Neural networks → non-convex. Many local minima and saddle points. GD finds *a* good minimum, not *the* best.

In [ ]:
fig = plt.figure(figsize=(14, 4))

x = np.linspace(-3, 3, 200)

ax1 = fig.add_subplot(1, 3, 1)
ax1.plot(x, x**2, 'C0')
ax1.set_title('Convex: one global min\n(linear/logistic regression)')
ax1.grid(alpha=0.3)

ax2 = fig.add_subplot(1, 3, 2)
ax2.plot(x, np.sin(2*x) + 0.2*x**2, 'C3')
ax2.set_title('Non-convex: many local minima\n(neural networks)')
ax2.grid(alpha=0.3)

# Saddle in 3D
ax3 = fig.add_subplot(1, 3, 3, projection='3d')
X, Y = np.meshgrid(np.linspace(-2, 2, 40), np.linspace(-2, 2, 40))
Zs = X**2 - Y**2
ax3.plot_surface(X, Y, Zs, cmap='coolwarm', alpha=0.8)
ax3.set_title('Saddle point\n(high-D neural nets)')

plt.tight_layout(); plt.show()

## 9. End-to-end: calculus on actual ML data

Let's fit a 1-feature linear regression on synthetic data by hand, using only calculus.

- Model: `ŷ = w · x + b`
- Loss: `L(w, b) = (1/m) Σ (ŷᵢ − yᵢ)²`   ← mean squared error, convex
- Gradients (chain rule on MSE):
  ```
  ∂L/∂w = (2/m) Σ (ŷᵢ − yᵢ) · xᵢ
  ∂L/∂b = (2/m) Σ (ŷᵢ − yᵢ)
  ```

This is what `LinearRegression` and `SGDRegressor` do internally. Watch the `w` and `b` converge to the true values.

In [ ]:
# Generate data from the true line y = 2x + 1 + noise
rng = np.random.default_rng(0)
m = 100
X = rng.uniform(-2, 2, m)
y_true_line = 2 * X + 1
y = y_true_line + rng.normal(scale=0.5, size=m)

plt.figure(figsize=(6, 4))
plt.scatter(X, y, alpha=0.6, label='data')
plt.plot(X, y_true_line, 'k--', label='true line y = 2x + 1')
plt.legend(); plt.grid(alpha=0.3); plt.title('Training data')
plt.show()

In [ ]:
# Gradient descent on MSE loss — calculus in action
w, b = 0.0, 0.0
eta = 0.05
history = []

for step in range(100):
    y_pred = w * X + b
    error = y_pred - y
    loss = (error**2).mean()

    # Gradients from the chain rule
    grad_w = (2 / m) * (error * X).sum()
    grad_b = (2 / m) * error.sum()

    # Gradient descent update
    w -= eta * grad_w
    b -= eta * grad_b

    history.append((w, b, loss))

history = np.array(history)
print(f'Final  w = {w:.3f}  (true: 2.0)')
print(f'Final  b = {b:.3f}  (true: 1.0)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history[:, 2]); axes[0].set_title('Loss over iterations'); axes[0].set_xlabel('step'); axes[0].set_ylabel('MSE'); axes[0].grid(alpha=0.3)

axes[1].scatter(X, y, alpha=0.4, label='data')
for i in [0, 5, 20, 99]:
    w_i, b_i, _ = history[i]
    axes[1].plot(X, w_i * X + b_i, label=f'step {i}')
axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_title('Fitted line evolves toward the data')
plt.tight_layout(); plt.show()

What happened here, in calculus terms:

1. Picked a loss `L(w, b)` measuring how wrong we are.
2. Took its gradient `[∂L/∂w, ∂L/∂b]` — slope of the loss surface in weight space.
3. Stepped against the gradient — slid downhill.
4. Loss converged to ~0.25 (the noise floor), and `(w, b)` converged to `(2, 1)` — the true generating parameters.

**This is every ML training loop in miniature.**

## 10. Deferred — come back when a chapter needs it

| Topic | When it becomes relevant |
|-------|--------------------------|
| Jacobian (matrix of partial derivatives) | Chapter on neural networks — needed if you derive backprop yourself. Otherwise autograd handles it. |
| Hessian (second derivatives) | Advanced optimizers (Newton's method). Most ML uses first-order methods — skip. |
| Taylor series | Theory of optimization — optional. |
| Integrals | Probability densities (Ch. 4 tiny bits, Bayesian methods). Almost never needed operationally. |
| Limits, continuity, proofs | Pure math — safe to skip. |

## 11. Calculus ⇄ ML concept map

| Calculus | ML equivalent |
|----------|---------------|
| Function `f(x)` | Model prediction `ŷ = f(x; w)` |
| Input variable `x` | Feature vector |
| Parameters of `f` | Learned weights `w` and bias `b` |
| Derivative / slope | "How much does the output change if I nudge one input?" |
| Partial derivative | Sensitivity of the loss to one weight |
| Gradient | Vector of all partials — direction of steepest loss increase |
| Minimum of a function | Best weights (lowest loss) |
| Chain rule | Backpropagation |
| Convex function | Loss that GD can globally minimize (MSE, log-loss) |
| Non-convex function | Neural network loss — GD finds *a* good minimum |

## Minimum bar before moving on

You should be able to:

- [ ] Read `f'(x₀)` and picture the slope of the tangent line at that point.
- [ ] Read `∂L/∂w_i` and say "slope of the loss along weight `w_i`, others fixed."
- [ ] Read `∇L(w)` and see an arrow pointing uphill on the loss surface.
- [ ] Explain the update rule `w ← w − η · ∇L(w)` word by word.
- [ ] Know why small `η` = slow, large `η` = overshoot.
- [ ] Recognize that convex = one global min; non-convex = many local mins.
- [ ] Say "backprop is the chain rule applied layer-by-layer" and mean it.

That's the calculus you need for Ch. 3 – 10. The rest compounds as the book demands it.